# Phase 3.2 : Entraînement de la tête uniquement

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase3_1_mobilenet_freezing.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1, 1.2 et 3.1 (téléchargement, organisation, pipeline MobileNetV2, base gelée + tête custom), sans les explications déjà vues.

## Reprise (setup phases 1.1, 1.2 et 3.1, condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


@tf.function
def _try_decode(img_bytes):
    # Force l'execution en mode graphe (via tf.function), qui est plus stricte que
    # l'execution eager sur certains formats (ex. BMP mal forme deguise en .jpg) et
    # reproduit exactement le mode d'execution utilise par image_dataset_from_directory
    # pendant l'entrainement.
    return tf.io.decode_image(img_bytes, channels=3, expand_animations=False)


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ni tf.io.decode_image() en
    # mode eager ne suffisent : ce dataset contient des fichiers que seul le decodeur en
    # mode graphe rejette (InvalidArgumentError, "Unknown image file format" ou erreurs
    # de dimension BMP). On teste donc chaque fichier via une tf.function.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            _try_decode(img_bytes)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
IMG_SIZE_TL = (160, 160)
BATCH_SIZE = 32

train_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)

val_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)

# preprocess_input de MobileNetV2 normalise dans [-1, 1] (pas [0, 1]) : pas de
# Rescaling(1./255) ici, preprocess_input le remplace.
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_tl = train_ds_tl.map(lambda x, y: (preprocess_input(x), y))
val_ds_tl = val_ds_tl.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds_tl = train_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_tl = val_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False)  # training=False : BatchNorm reste en mode inference
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_tl = tf.keras.Model(inputs, outputs)

In [ ]:
import matplotlib.pyplot as plt


def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Val loss")
    ax1.set_title(f"{title} - Loss")
    ax1.legend()

    ax2.plot(history.history["accuracy"], label="Train accuracy")
    ax2.plot(history.history["val_accuracy"], label="Val accuracy")
    ax2.set_title(f"{title} - Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"curves_{title.lower().replace(' ', '_').replace('+', '')}.png", dpi=100)
    plt.show()

## Phase 3.2 : Entraînement de la tête uniquement

**Objectif** : entraîner uniquement la tête de classification (base gelée) pendant 10 epochs, observer la convergence rapide, et vérifier que la `val_accuracy` dépasse 90%.

In [ ]:
import datetime, time

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

log_dir_tl = "logs/transfer/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback_tl = tf.keras.callbacks.TensorBoard(log_dir=log_dir_tl, histogram_freq=1)

start = time.time()

history_tl_head = model_tl.fit(
    train_ds_tl,
    epochs=10,
    validation_data=val_ds_tl,
    callbacks=[tensorboard_callback_tl],
)

training_time_head = time.time() - start
print(f"Temps d'entrainement (tete seule) : {training_time_head:.0f}s")
print(f"val_accuracy finale (tete seule) : {max(history_tl_head.history['val_accuracy']):.3f}")

In [ ]:
plot_history(history_tl_head, "Transfer Learning - tete seule")

In [ ]:
import json as _json

with open("history_tl_head.json", "w") as f:
    _json.dump(history_tl_head.history, f)
print("history_tl_head.json sauvegarde.")

### Vérifications avant la phase 3.3

- **Happy path** : `val_accuracy` dépasse 90% en 3-5 epochs, convergence visiblement plus rapide que TP1/TP2. Train et val convergent ensemble, gap quasi inexistant — normal, la base ne bouge pas, seule la tête (peu de paramètres) s'adapte.
- **Edge case** : entraîner avec `learning_rate=1e-1` au lieu de `1e-3` ferait osciller violemment la `val_accuracy` — la tête bouge trop à chaque batch.
- **Adversarial** : oublier `base_model.trainable = False` (déjà fait en phase 3.1, mais à vérifier si tu modifies le code) ferait s'entraîner toute la base avec `lr=1e-3` — le pretrained se détériore en 2-3 epochs, la `val_accuracy` redescendrait sous les résultats de TP1.

**Prochaine étape** : Phase 3.3 — dégeler les dernières couches de MobileNetV2 pour du fine-tuning.